### ЗАДАЧА: Панель расчета и выпуска выплат продавцам по паттерну `MVC`

Команда finance operations обрабатывает кейсы по расчету выплат продавцам маркетплейса в конце периода.
Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `seller` — продавец;
- `period` — расчетный период;
- `gross_sales` — валовая выручка продавца;
- `returns_amount` — сумма возвратов за период;
- `ads_spend` — расходы на рекламу;
- `previous_debt` — долг с прошлых периодов;
- `commission_rate` — комиссия маркетплейса;
- `payout_cap` — лимит на единовременную выплату;
- `commission_amount` — сумма комиссии;
- `preliminary_payout` — предварительная выплата до лимита;
- `holdback_amount` — сумма удержания из-за лимита;
- `net_payout` — финальная сумма к выплате;
- `status` — текущий статус кейса;
- `analyst` — сотрудник, который ведет кейс;
- `approved` — согласован ли расчет;
- `decision` — итоговое решение.

### Формулы

При создании кейса и после любого пересчета нужно правильно считать:
- `commission_amount = gross_sales * commission_rate`
- `preliminary_payout = gross_sales - commission_amount - returns_amount - ads_spend - previous_debt`
- `holdback_amount = max(preliminary_payout - payout_cap, 0.0)`
- `net_payout = preliminary_payout - holdback_amount`
- все числовые результаты нужно округлять до 2 знаков.

### Сводка

Нужен summary-метод, который возвращает:
- количество кейсов по статусам;
- `total_net_payout` — суммарный `net_payout` по всем кейсам;
- `total_holdback` — суммарный `holdback_amount`;
- `total_negative_preliminary` — сумма модулей кейсов, где `preliminary_payout < 0`;
- `released_total` — сумма `net_payout` по кейсам со статусом `released`.

### Статусы кейса

- `new`
- `reviewing`
- `recalculated`
- `ready_to_release`
- `released`
- `disputed`
- `escalated`

### Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `analyst` несуществующему кейсу;
- финальные кейсы (`released`, `disputed`, `escalated`) нельзя менять дальше;
- начать review можно только из `new` и только если назначен `analyst`;
- применить корректировку можно только из `reviewing` или `recalculated`;
- при корректировке нельзя допустить отрицательные значения у `gross_sales`, `returns_amount`, `ads_spend`, `previous_debt`, `payout_cap`;
- после корректировки нужно пересчитать все вычисляемые поля и перевести кейс в `recalculated`;
- согласовать расчет можно только из `reviewing` или `recalculated`;
- перевод в `ready_to_release` возможен только из `reviewing` или `recalculated`;
- перевод в `ready_to_release` возможен только если `approved == True`;
- перевод в `ready_to_release` невозможен, если `preliminary_payout < 0`;
- выпустить выплату можно только из `ready_to_release`;
- выпустить выплату можно только если `net_payout > 0`;
- перевести кейс в `disputed` можно только из `reviewing`, `recalculated` или `ready_to_release`;
- перевести кейс в `escalated` можно только из `reviewing`, `recalculated` или `ready_to_release`;
- при финальных статусах нужно записывать `decision`.

In [1]:
from dataclasses import dataclass


initial_cases = [
    ("SS-100", "best-gadgets", "2026-03", 25000.0, 3000.0, 1200.0, 1500.0, 0.12, 15000.0),
    ("SS-101", "home-decor-plus", "2026-03", 7000.0, 2000.0, 900.0, 1800.0, 0.12, 5000.0),
]

actions = [
    ("show",),
    ("review", "SS-100"),
    ("assign", "SS-100", "Olga"),
    ("review", "SS-100"),
    ("ready", "SS-100"),
    ("approve", "SS-100"),
    ("ready", "SS-100"),
    ("release", "SS-100", "released_to_wallet"),
    ("create", "SS-102", "sport-zone", "2026-03", 9000.0, 500.0, 1000.0, 2000.0, 0.15, 4000.0),
    ("assign", "SS-102", "Max"),
    ("review", "SS-102"),
    ("adjust", "SS-102", -2000.0, 1500.0, 0.0, 500.0, 0.0),
    ("approve", "SS-102"),
    ("ready", "SS-102"),
    ("dispute", "SS-102", "seller_disagrees"),
    ("create", "SS-103", "city-market", "2026-03", 5000.0, 800.0, 400.0, 3200.0, 0.14, 6000.0),
    ("assign", "SS-103", "Ina"),
    ("review", "SS-103"),
    ("approve", "SS-103"),
    ("ready", "SS-103"),
    ("escalate", "SS-103", "negative_payout"),
    ("show",),
]


@dataclass
class SettlementCase:
    case_id: str
    seller: str
    period: str
    gross_sales: float
    returns_amount: float
    ads_spend: float
    previous_debt: float
    commission_rate: float
    payout_cap: float
    commission_amount: float
    preliminary_payout: float
    holdback_amount: float
    net_payout: float
    status: str = "new"
    analyst: str | None = None
    approved: bool = False
    decision: str | None = None


class SettlementModel:
    def __init__(self):
        self.cases = {}
        self.done_statuses = {'released','disputed','escalated'}

    def _calculate_commission_amount(self, gross_sales: float, commission_rate: float) -> float:
        return round(gross_sales * commission_rate, 2)

    def _calculate_preliminary_payout(self, gross_sales: float, commission_amount: float, returns_amount: float, ads_spend: float, previous_debt: float) -> float:
        return round(gross_sales - commission_amount - returns_amount - ads_spend - previous_debt, 2)

    def _calculate_holdback_amount(self, preliminary_payout: float, payout_cap: float) -> float:
        if preliminary_payout < payout_cap:
            return round(max(preliminary_payout - payout_cap, 0.0),2)
        return 0.0

    def _calculate_net_payout(self, preliminary_payout: float, holdback_amount: float) -> float:
        return round(preliminary_payout - holdback_amount, 2)

    def _build_case(self, case_id: str, seller: str, period: str, gross_sales: float, returns_amount: float, ads_spend: float, previous_debt: float, commission_rate: float, payout_cap: float) -> SettlementCase:
        
        commission_amount = self._calculate_commission_amount(gross_sales, commission_rate)
        preliminary_payout = self._calculate_preliminary_payout(gross_sales, commission_amount, returns_amount, ads_spend, previous_debt)
        holdback_amount = self._calculate_holdback_amount(preliminary_payout, payout_cap)
        net_payout = self._calculate_net_payout(preliminary_payout, holdback_amount)
        return SettlementCase(case_id, seller, period, gross_sales, returns_amount, ads_spend, previous_debt, commission_rate, payout_cap, commission_amount, preliminary_payout, holdback_amount, net_payout)

    def add_case(self, case_id: str, seller: str, period: str, gross_sales: float, returns_amount: float, ads_spend: float, previous_debt: float, commission_rate: float, payout_cap: float) -> None:
        if case_id in self.cases:
            raise ValueError('already exists')
        self.cases[case_id] = self._build_case(case_id, seller, period, gross_sales, returns_amount, ads_spend, previous_debt, commission_rate, payout_cap)

    def assign_analyst(self, case_id: str, analyst: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')

        self.cases[case_id].analyst = analyst

    def start_review(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        
        if self.cases[case_id].status != 'new':
            raise ValueError('not new')
        if self.cases[case_id].analyst is None:
            raise ValueError('no analyst')
        
        self.cases[case_id].status = "reviewing"

    def apply_adjustment(self, case_id: str, gross_delta: float, returns_delta: float, ads_delta: float, debt_delta: float, payout_cap_delta: float) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status not in {'reviewing','recalculated'}:
            raise ValueError('not reviewing,recalculated')

        case = self.cases[case_id]
        case.gross_sales += gross_delta
        case.returns_amount += returns_delta
        case.ads_spend += ads_delta
        case.previous_debt += debt_delta
        case.payout_cap += payout_cap_delta
        case.commission_amount = self._calculate_commission_amount(case.gross_sales, case.commission_rate)
        case.preliminary_payout = self._calculate_preliminary_payout(case.gross_sales, case.commission_amount, case.returns_amount, case.ads_spend, case.previous_debt)
        case.holdback_amount = self._calculate_holdback_amount(case.preliminary_payout, case.payout_cap)
        case.net_payout = self._calculate_net_payout(case.preliminary_payout, case.holdback_amount)
        self.cases[case_id].status = 'recalculated'

    def approve_case(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status not in {'reviewing','recalculated'}:
            raise ValueError('not reviewing,recalculated')

        
        self.cases[case_id].approved = True

    def mark_ready_to_release(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].approved != True:
            raise ValueError('not True')
        if self.cases[case_id].preliminary_payout < 0:
            raise ValueError('preliminary_payout < 0') 
        
        self.cases[case_id].status = "ready_to_release"

    def release_case(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status != "ready_to_release":
            raise ValueError('not ready to release')
        if self.cases[case_id].net_payout <= 0:
            raise ValueError('net_payout <= 0')
        case = self.cases[case_id]
        case.status = "released"
        case.decision = decision

    def dispute_case(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status not in {'reviewing','recalculated','ready_to_release'}:
            raise ValueError('cant dispute not reviewing,recalculated,ready_to_release')
        case = self.cases[case_id]
        case.status = "disputed"
        case.decision = decision

    def escalate_case(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status not in {'reviewing','recalculated','ready_to_release'}:
            raise ValueError('cant escalate not reviewing,recalculated,ready_to_release')

        case = self.cases[case_id]
        case.status = "escalated"
        case.decision = decision

    def list_cases(self) -> list[str]:
        rows = []
        for case in self.cases.values():
            rows.append(
                f"{case.case_id} | {case.seller} | {case.period} | {case.gross_sales} | {case.returns_amount} | {case.ads_spend} | "
                f"{case.previous_debt} | {case.commission_amount} | {case.preliminary_payout} | {case.holdback_amount} | {case.net_payout} | "
                f"{case.status} | {case.analyst} | {case.approved} | {case.decision}"
            )
        return rows

    def summary(self) -> dict[str, float | int]:
        result = {
            "total_cases": 0,
            "total_net_payout": 0.0,
            "total_holdback": 0.0,
            "total_negative_preliminary": 0.0,
            "released_total": 0.0,
        }
        for case in self.cases.values():
            result[case.status] = result.get(case.status, 0) + 1
            result["total_cases"] += 1
            result["total_net_payout"] += case.preliminary_payout
            result["total_holdback"] += case.net_payout
            if case.preliminary_payout < 0:
                result["total_negative_preliminary"] += case.preliminary_payout
            if case.status == "released":
                result["released_total"] += case.holdback_amount
        return result


class SettlementView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        print("Settlement cases:")
        for row in rows:
            print(row)

    @staticmethod
    def render_summary(summary: dict[str, float | int]) -> None:
        print("Summary:", summary)

    @staticmethod
    def render_success(message: str) -> None:
        print("SUCCESS:", message)

    @staticmethod
    def render_error(message: str) -> None:
        print("ERROR:", message)


class SettlementController:
    def __init__(self, model: SettlementModel, view: SettlementView):
        self.model = model
        self.view = view

    def create_case(self, case_id: str, seller: str, period: str, gross_sales: float, returns_amount: float, ads_spend: float, previous_debt: float, commission_rate: float, payout_cap: float) -> None:
        try:
            self.model.add_case(case_id, seller, period, gross_sales, returns_amount, ads_spend, previous_debt, commission_rate, payout_cap)
            self.view.render_success(f"Case {case_id} created")
        except ValueError as error:
            self.view.render_error(str(error))

    def assign_analyst(self, case_id: str, analyst: str) -> None:
        try:
            self.model.assign_analyst(case_id, analyst)
            self.view.render_success(f"Analyst assigned to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def start_review(self, case_id: str) -> None:
        try:
            self.model.start_review(case_id)
            self.view.render_success(f"Case {case_id} moved to review")
        except ValueError as error:
            self.view.render_error(str(error))

    def apply_adjustment(self, case_id: str, gross_delta: float, returns_delta: float, ads_delta: float, debt_delta: float, payout_cap_delta: float) -> None:
        try:
            self.model.apply_adjustment(case_id, gross_delta, returns_delta, ads_delta, debt_delta, payout_cap_delta)
            self.view.render_success(f"Adjustment applied to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def approve_case(self, case_id: str) -> None:
        try:
            self.model.approve_case(case_id)
            self.view.render_success(f"Case {case_id} approved")
        except ValueError as error:
            self.view.render_error(str(error))

    def mark_ready_to_release(self, case_id: str) -> None:
        try:
            self.model.mark_ready_to_release(case_id)
            self.view.render_success(f"Case {case_id} is ready to release")
        except ValueError as error:
            self.view.render_error(str(error))

    def release_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.release_case(case_id, decision)
            self.view.render_success(f"Case {case_id} released")
        except ValueError as error:
            self.view.render_error(str(error))

    def dispute_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.dispute_case(case_id, decision)
            self.view.render_success(f"Case {case_id} disputed")
        except ValueError as error:
            self.view.render_error(str(error))

    def escalate_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.escalate_case(case_id, decision)
            self.view.render_success(f"Case {case_id} escalated")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        self.view.render_cases(self.model.list_cases())
        self.view.render_summary(self.model.summary())


model = SettlementModel()
view = SettlementView()
controller = SettlementController(model, view)

for case_id, seller, period, gross_sales, returns_amount, ads_spend, previous_debt, commission_rate, payout_cap in initial_cases:
    model.add_case(case_id, seller, period, gross_sales, returns_amount, ads_spend, previous_debt, commission_rate, payout_cap)

for action in actions:
    if action[0] == "show":
        controller.show_cases()
    elif action[0] == "create":
        _, case_id, seller, period, gross_sales, returns_amount, ads_spend, previous_debt, commission_rate, payout_cap = action
        controller.create_case(case_id, seller, period, gross_sales, returns_amount, ads_spend, previous_debt, commission_rate, payout_cap)
    elif action[0] == "assign":
        _, case_id, analyst = action
        controller.assign_analyst(case_id, analyst)
    elif action[0] == "review":
        _, case_id = action
        controller.start_review(case_id)
    elif action[0] == "adjust":
        _, case_id, gross_delta, returns_delta, ads_delta, debt_delta, payout_cap_delta = action
        controller.apply_adjustment(case_id, gross_delta, returns_delta, ads_delta, debt_delta, payout_cap_delta)
    elif action[0] == "approve":
        _, case_id = action
        controller.approve_case(case_id)
    elif action[0] == "ready":
        _, case_id = action
        controller.mark_ready_to_release(case_id)
    elif action[0] == "release":
        _, case_id, decision = action
        controller.release_case(case_id, decision)
    elif action[0] == "dispute":
        _, case_id, decision = action
        controller.dispute_case(case_id, decision)
    elif action[0] == "escalate":
        _, case_id, decision = action
        controller.escalate_case(case_id, decision)

print()
print("Финальное состояние:")
controller.show_cases()

Settlement cases:
SS-100 | best-gadgets | 2026-03 | 25000.0 | 3000.0 | 1200.0 | 1500.0 | 3000.0 | 16300.0 | 0.0 | 16300.0 | new | None | False | None
SS-101 | home-decor-plus | 2026-03 | 7000.0 | 2000.0 | 900.0 | 1800.0 | 840.0 | 1460.0 | 0.0 | 1460.0 | new | None | False | None
Summary: {'total_cases': 2, 'total_net_payout': 17760.0, 'total_holdback': 17760.0, 'total_negative_preliminary': 0.0, 'released_total': 0.0, 'new': 2}
ERROR: no analyst
SUCCESS: Analyst assigned to SS-100
SUCCESS: Case SS-100 moved to review
ERROR: not True
SUCCESS: Case SS-100 approved
SUCCESS: Case SS-100 is ready to release
SUCCESS: Case SS-100 released
SUCCESS: Case SS-102 created
SUCCESS: Analyst assigned to SS-102
SUCCESS: Case SS-102 moved to review
SUCCESS: Adjustment applied to SS-102
SUCCESS: Case SS-102 approved
SUCCESS: Case SS-102 is ready to release
SUCCESS: Case SS-102 disputed
SUCCESS: Case SS-103 created
SUCCESS: Analyst assigned to SS-103
SUCCESS: Case SS-103 moved to review
SUCCESS: Case SS-

In [ ]:
# Settlement cases:
# SS-100 | best-gadgets | 2026-03 | 25000.0 | 3000.0 | 1200.0 | 1500.0 | 3000.0 | 16300.0 | 1300.0 | 15000.0 | new | None | False | None
# SS-101 | home-decor-plus | 2026-03 | 7000.0 | 2000.0 | 900.0 | 1800.0 | 840.0 | 1460.0 | 0.0 | 1460.0 | new | None | False | None
# Summary: {'total_cases': 2, 'total_net_payout': 16460.0, 'total_holdback': 1300.0, 'total_negative_preliminary': 0.0, 'released_total': 0.0, 'new': 2}
# ERROR: Analyst is required
# SUCCESS: Analyst assigned to SS-100
# SUCCESS: Case SS-100 moved to review
# ERROR: Case is not approved
# SUCCESS: Case SS-100 approved
# SUCCESS: Case SS-100 is ready to release
# SUCCESS: Case SS-100 released
# SUCCESS: Case SS-102 created
# SUCCESS: Analyst assigned to SS-102
# SUCCESS: Case SS-102 moved to review
# SUCCESS: Adjustment applied to SS-102
# SUCCESS: Case SS-102 approved
# SUCCESS: Case SS-102 is ready to release
# SUCCESS: Case SS-102 disputed
# SUCCESS: Case SS-103 created
# SUCCESS: Analyst assigned to SS-103
# SUCCESS: Case SS-103 moved to review
# SUCCESS: Case SS-103 approved
# ERROR: Preliminary payout is negative
# SUCCESS: Case SS-103 escalated
# Settlement cases:
# SS-100 | best-gadgets | 2026-03 | 25000.0 | 3000.0 | 1200.0 | 1500.0 | 3000.0 | 16300.0 | 1300.0 | 15000.0 | released | Olga | True | released_to_wallet
# SS-101 | home-decor-plus | 2026-03 | 7000.0 | 2000.0 | 900.0 | 1800.0 | 840.0 | 1460.0 | 0.0 | 1460.0 | new | None | False | None
# SS-102 | sport-zone | 2026-03 | 7000.0 | 2000.0 | 1000.0 | 2500.0 | 1050.0 | 450.0 | 0.0 | 450.0 | disputed | Max | True | seller_disagrees
# SS-103 | city-market | 2026-03 | 5000.0 | 800.0 | 400.0 | 3200.0 | 700.0 | -100.0 | 0.0 | -100.0 | escalated | Ina | True | negative_payout
# Summary: {'total_cases': 4, 'total_net_payout': 16810.0, 'total_holdback': 1300.0, 'total_negative_preliminary': 100.0, 'released_total': 15000.0, 'released': 1, 'new': 1, 'disputed': 1, 'escalated': 1}
#
# Финальное состояние:
# Settlement cases:
# SS-100 | best-gadgets | 2026-03 | 25000.0 | 3000.0 | 1200.0 | 1500.0 | 3000.0 | 16300.0 | 1300.0 | 15000.0 | released | Olga | True | released_to_wallet
# SS-101 | home-decor-plus | 2026-03 | 7000.0 | 2000.0 | 900.0 | 1800.0 | 840.0 | 1460.0 | 0.0 | 1460.0 | new | None | False | None
# SS-102 | sport-zone | 2026-03 | 7000.0 | 2000.0 | 1000.0 | 2500.0 | 1050.0 | 450.0 | 0.0 | 450.0 | disputed | Max | True | seller_disagrees
# SS-103 | city-market | 2026-03 | 5000.0 | 800.0 | 400.0 | 3200.0 | 700.0 | -100.0 | 0.0 | -100.0 | escalated | Ina | True | negative_payout
# Summary: {'total_cases': 4, 'total_net_payout': 16810.0, 'total_holdback': 1300.0, 'total_negative_preliminary': 100.0, 'released_total': 15000.0, 'released': 1, 'new': 1, 'disputed': 1, 'escalated': 1}